<a href="https://colab.research.google.com/github/oliveirasamuel5959/VRDFormer_VRD/blob/main/notebooks/colab_kaggle_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VRDFormer Training on Colab T4 with Kaggle Data

This notebook trains the full VRDFormer pipeline (Stage 1 + Stage 2) on VidOR using a Colab T4 GPU.
Data is hosted on Kaggle as a dataset — no Google Drive space needed.

**Prerequisites:**
- GitHub fork of VRDFormer_VRD with Kaggle configs + bug fixes
- Kaggle dataset uploaded (contains data/vidor/ + metadata + DETR weights)
- Kaggle API key (kaggle.json) from kaggle.com/settings → API → Create New Token

https://colab.research.google.com/github/oliveirasamuel5959/VRDFormer_VRD/blob/main/notebooks/colab_kaggle_train.ipynb

## 1. Clone Repo & Install Dependencies

In [ ]:
# Clone your fork (replace YOUR_USERNAME)
!git clone https://github.com/oliveirasamuel5959/VRDFormer_VRD.git
%cd VRDFormer_VRD

# Install dependencies (PyTorch + torchvision pre-installed on Colab T4)
!pip install decord timm scipy lap -q
!pip install -U 'git+https://github.com/timmeinhardt/cocoapi.git#subdirectory=PythonAPI' -q

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Number of available GPUs: {torch.cuda.device_count()}')

Cloning into 'VRDFormer_VRD'...
remote: Enumerating objects: 238, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 238 (delta 27), reused 28 (delta 18), pack-reused 181 (from 1)
Receiving objects: 100% (238/238), 252.54 KiB | 1.24 MiB/s, done.
Resolving deltas: 100% (92/92), done.
/content/VRDFormer_VRD
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 129.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 91.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
PyTorch: 2.11.0+cu128
CUDA available: True


## 2. Download Kaggle Dataset

In [3]:
# Upload your kaggle.json API key
from google.colab import files
files.upload()  # Select kaggle.json from your local machine

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!pip install kaggle -q

# Make data directory with metadata and weights
!mkdir -p data/metadata data/weights data/ckpts

# Download and extract (replace YOUR_KAGGLE_USERNAME)
!kaggle datasets download samuelpatricio/vrdformer-vidor
!unzip -q vrdformer-vidor.zip -d data
!rm vrdformer-vidor.zip



Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/samuelpatricio/vrdformer-vidor
License(s): unknown
100% 27.4G/27.4G [22:45<00:00, 21.5MB/s]



In [4]:
# Move bundled metadata and weights into place
!mv data/vidor/metadata/* data/metadata/ 2>/dev/null || echo 'metadata already in place'
!mv data/vidor/detr-r101-2c7b67e5.pth data/weights/ 2>/dev/null || echo 'weights already in place'

print('Dataset ready.')

Dataset ready.


## 3. Verify GPU & Data

In [5]:
!nvidia-smi

import os
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print()

checks = [
    ('data/vidor/videos', 'dir'),
    ('data/vidor/annotations/train', 'dir'),
    ('data/vidor/annotations/val', 'dir'),
    ('data/metadata/vidor_train_frames_stage1.json', 'file'),
    ('data/metadata/vidor_val_frames.json', 'file'),
    ('data/weights/detr-r101-2c7b67e5.pth', 'file'),
    ('configs/vidor_kaggle_stage1.json', 'file'),
    ('configs/vidor_kaggle_stage2.json', 'file'),
]

all_ok = True
for path, kind in checks:
    if kind == 'dir':
        exists = os.path.isdir(path)
    else:
        exists = os.path.isfile(path)
    size = ''
    if exists and kind == 'file':
        size = f' ({os.path.getsize(path) / 1e6:.1f} MB)'
    print(f'  {"✅" if exists else "❌"} {path}{size}')
    if not exists:
        all_ok = False

if all_ok:
    print('\nAll checks passed — ready to train!')
else:
    print('\n⚠️  Some files missing — check the Kaggle dataset download.')

Wed Aug  5 12:46:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             44W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 4. Smoke Test (Optional)

Run 1 epoch on 100 videos to verify the pipeline works before committing 5+ hours to full training.

In [6]:
# # Smoke test — Running to verify the fix
# !python main.py \
#     --accumulate_steps 1 \
#     --lr_backbone 1e-5 \
#     --lr 5e-5 \
#     --num_queries 200 \
#     --dataset_config configs/vidorpart_local_stage1.json \
#     --epochs 1

## 5. Stage 1 Training

Pair detection + tracking. **~4-6 hours**, uses ~13-15 GB VRAM.

**What to watch:**
- Loss: ~2.5 → ~0.9 over 5 epochs
- sub_class_acc: ~30% → ~88%
- Checkpoints saved to `data/ckpts/vidor_stage1/`
- *Known issue:* NameError at epoch end is harmless (Stage 1 has no eval path)

In [7]:
%%time
!python main.py \
    --accumulate_steps 1 \
    --lr_backbone 1e-5 \
    --lr 5e-5 \
    --num_queries 200 \
    --dataset_config configs/vidor_kaggle_stage1.json

Not using distributed mode
git:
  sha: 1e92569ef7281d43d24b6b506efd7c82719db6d0, status: clean, branch: main

Namespace(lr=5e-05, lr_backbone=1e-05, lr_drop=4, weight_decay=0.0001, batch_size=2, epochs=5, optimizer='adam', clip_max_norm=0.1, accumulate_steps=1.0, eval_skip=1, debug=False, vis_and_log_interval=10, schedule='linear_with_warmup', ema=False, ema_decay=0.9998, fraction_warmup_steps=0.01, frozen_weights=None, roi_pool_type='avg', backbone='resnet101', dilation=False, position_embedding='sine_3d_v2', enc_layers=6, dec_layers=6, dim_feedforward=2048, hidden_dim=256, dropout=0.1, nheads=8, num_queries=200, pre_norm=False, aux_loss=False, eval=False, eval_mode='evalGT', output_dir='data/ckpts/vidor_stage1', device='cuda', seed=42, resume='', resume_shift_neuron=False, pretrain='data/weights/detr-r101-2c7b67e5.pth', start_epoch=0, num_workers=2, dataset_config='configs/vidor_kaggle_stage1.json', world_size=1, dist_url='env://', local_rank=-1, stage=1, coco_path='', vidvrd_path=''

### Verify Stage 1 Output

In [ ]:
import os, glob
ckpts = sorted(glob.glob('data/ckpts/vidor_stage1/checkpoint*.pth'))
print(f'Stage 1 checkpoints: {len(ckpts)}')
for ckpt in ckpts:
    size_mb = os.path.getsize(ckpt) / 1e6
    print(f'  {ckpt:<55} {size_mb:.1f} MB')

Stage 1 checkpoints: 0


## 6. Stage 2 Training

Temporal relation classification. **~1-2 hours**, uses ~10-12 GB VRAM.

Loads `checkpoint0004.pth` from Stage 1. Uses GT boxes via ROI Align (no box regression).

In [ ]:
%%time
!python main.py \
    --accumulate_steps 1 \
    --lr_backbone 1e-5 \
    --lr 5e-5 \
    --num_queries 200 \
    --dataset_config configs/vidor_kaggle_stage2.json

## 7. Evaluation

Evaluates on VidOR validation set. Reports detection mAP, recall@50/100, and tagging precision@1/5/10 — each under overall, zero-shot, and generalized-zero-shot settings.

In [ ]:
!python main.py \
    --eval \
    --dataset_config configs/vidor_kaggle_stage2.json \
    --resume data/ckpts/vidor_stage2/checkpoint.pth

## 8. Download Checkpoints

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# %cd /content/VRDFormer_VRD/

/content/VRDFormer_VRD


In [ ]:
# Copy the zip file from Drive to local Colab machine
# !mkdir -p data/vidor/ data/metadata/

In [ ]:
# !cp -r /content/drive/MyDrive/data/* data/vidor

In [ ]:
# %cd /content/VRDFormer_VRD/data

/content/VRDFormer_VRD/data


In [ ]:
# %%time
# !python prepare.py --func get_anno --dbname vidor --root_dir .

100% 7835/7835 [11:49<00:00, 11.04it/s]
CPU times: user 3.9 s, sys: 741 ms, total: 4.64 s
Wall time: 16min 3s


In [ ]:
# !zip -r metadata.zip metadata

updating: metadata/ (stored 0%)
updating: metadata/vidor_annotations.pkl (deflated 92%)


In [ ]:
# from google.colab import files
# files.download("metadata.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Zip and download checkpoints to your local machine
!zip -r checkpoints.zip data/ckpts/
from google.colab import files
files.download('checkpoints.zip')

print('\nDone! Key files:')
print('  Stage 1: data/ckpts/vidor_stage1/checkpoint0004.pth')
print('  Stage 2: data/ckpts/vidor_stage2/checkpoint.pth')

In [ ]:
# !git restore engine.py

In [2]:
%cd /content/VRDFormer_VRD
!git pull origin main

/content/VRDFormer_VRD
From https://github.com/oliveirasamuel5959/VRDFormer_VRD
 * branch            main       -> FETCH_HEAD
Already up to date.


## Colab Keep-Alive

Colab disconnects after ~90 min of inactivity. Paste this in the browser console (F12) to prevent disconnects:

```javascript
function ClickConnect(){
    document.querySelector("colab-connect-button").click()
}
setInterval(ClickConnect, 60000)
```